# 07 — Visualization

**Purpose.** Turn `06`'s correlation table into figures — useful regardless
of whether the go/no-go verdict comes back GO or NO-GO; you want to *see*
the relationship either way, not just read a correlation coefficient.

**No new formulas or decisions here** — purely presentation of numbers
already computed and saved by `05`/`06`.

**Expected runtime:** a few seconds.
**GPU:** not used.


## Step 1 — Locate project, load the correlation table (`06`)

In [ ]:
import sys
from pathlib import Path

if "PROJECT_ROOT" not in dir():
    _here = Path.cwd()
    for candidate in [_here, *_here.parents]:
        if (candidate / "src" / "utils" / "env_utils.py").exists():
            PROJECT_ROOT = candidate
            break
    else:
        raise FileNotFoundError("PROJECT_ROOT not found. Run 00_environment.ipynb first.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pickle
import matplotlib.pyplot as plt
import numpy as np

MIXTURE_SIZE = 1
outputs_dir = PATHS["outputs"] if "PATHS" in dir() else PROJECT_ROOT / "outputs"
figures_dir = PATHS["figures"] if "PATHS" in dir() else PROJECT_ROOT / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

with open(outputs_dir / f"uci_pilot_correlation_mixture_{MIXTURE_SIZE}.pkl", "rb") as f:
    saved = pickle.load(f)
df = saved["df"]
correlations = saved["correlations"]
print(df)


## Step 2 — NRC vs. PCE scatter plots (the core go/no-go figure)

A 2x2 grid: {NRC1, NRC2} x {PCE(BASE), PCE(BASE+QR)}, each panel annotated
with its Spearman rho/p from `06` — this is the single figure that most
directly communicates the go/no-go result.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 9))
pairs = [("nrc1", "pce_base"), ("nrc1", "pce_qrc"), ("nrc2", "pce_base"), ("nrc2", "pce_qrc")]

for ax, (x_col, y_col) in zip(axes.flat, pairs):
    ax.scatter(df[x_col], df[y_col], s=40, alpha=0.8, edgecolor="k", linewidth=0.5)
    for name, row in df.iterrows():
        ax.annotate(name, (row[x_col], row[y_col]), fontsize=7, alpha=0.7,
                    xytext=(3, 3), textcoords="offset points")
    rho, p, decision = correlations[(x_col, y_col)]
    ax.set_xlabel(x_col.upper())
    ax.set_ylabel(y_col.upper())
    ax.set_title(f"{x_col} vs {y_col}\nrho={rho:.2f}, p={p:.2f} -> {decision}", fontsize=10)
    ax.grid(alpha=0.3)

fig.suptitle(f"NRC-distance vs. PCE (n={len(df)} pilot datasets, mixture_size={MIXTURE_SIZE})", y=1.00)
fig.tight_layout()
fig.savefig(figures_dir / f"nrc_vs_pce_scatter_mixture_{MIXTURE_SIZE}.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {figures_dir / f'nrc_vs_pce_scatter_mixture_{MIXTURE_SIZE}.png'}")


## Step 3 — Does Quantile Recalibration actually help, per dataset?

A simple, honest before/after bar chart — independent of the NRC question,
this is worth checking on its own: QR should reduce PCE on (most) datasets.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(df))
width = 0.35
ax.bar(x - width/2, df["pce_base"], width, label="PCE(BASE)", color="#d62728", alpha=0.85)
ax.bar(x + width/2, df["pce_qrc"], width, label="PCE(BASE + QR)", color="#2ca02c", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(df.index, rotation=45, ha="right")
ax.set_ylabel("PCE (lower = better calibrated)")
ax.set_title("Effect of Quantile Recalibration per dataset")
ax.legend()
ax.grid(alpha=0.3, axis="y")
fig.tight_layout()
fig.savefig(figures_dir / f"pce_before_after_qr_mixture_{MIXTURE_SIZE}.png", dpi=150, bbox_inches="tight")
plt.show()

n_improved = (df["pce_qrc"] < df["pce_base"]).sum()
print(f"QR improved PCE on {n_improved}/{len(df)} datasets.")


## Step 4 — Reliability-diagram-style PIT histogram (one dataset, illustrative)

Shows what "PCE" actually looks like as a picture for a single dataset — a
uniform histogram means well-calibrated PITs (matching Section 2's
definition: `Z ~ Uniform(0,1)` for a calibrated model).

In [ ]:
from src.metrics import pce as pce_mod
from src.models import pilot_mixture_model as pmm
from src.utils import env_utils
import torch
import torch.nn.functional as F

EXTERNAL_DIR = PROJECT_ROOT / "external" / "quantile-recalibration-training"
if not EXTERNAL_DIR.exists():
    env_utils.clone_or_pull_repo(
        repo_url="https://github.com/Vekteur/quantile-recalibration-training.git",
        dest=EXTERNAL_DIR, branch="main",
    )
MixturePrediction = pmm.import_mixture_prediction(PROJECT_ROOT)
pmm.set_mixture_prediction_cls(MixturePrediction)

illustrative_dataset = df["pce_base"].idxmax()  # show the worst-calibrated one -- most informative
with open(outputs_dir / "uci_pilot_splits.pkl", "rb") as f:
    all_splits = pickle.load(f)["splits"]
ckpt_dir = (PATHS["checkpoints"] if "PATHS" in dir() else PROJECT_ROOT / "checkpoints") / f"mixture_{MIXTURE_SIZE}"
module = pmm.load_pilot_checkpoint(ckpt_dir / f"{illustrative_dataset}.pt")

x_test = torch.from_numpy(all_splits[illustrative_dataset]["test"]["x"]).to(torch.float32)
module.model.eval()
with torch.no_grad():
    means, rhos, _ = module.model(x_test)
    stds = F.softplus(rhos) + 1e-3
z_test = pce_mod.compute_pit_gaussian(
    means.numpy().ravel(), stds.numpy().ravel(), all_splits[illustrative_dataset]["test"]["y"],
)

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(z_test, bins=15, range=(0, 1), color="#1f77b4", edgecolor="white", alpha=0.85)
ax.axhline(len(z_test) / 15, color="k", linestyle="--", label="Perfectly uniform")
ax.set_xlabel("PIT value")
ax.set_ylabel("Count")
ax.set_title(f"PIT histogram, {illustrative_dataset} (PCE={df.loc[illustrative_dataset, 'pce_base']:.3f})")
ax.legend()
fig.tight_layout()
fig.savefig(figures_dir / f"pit_histogram_{illustrative_dataset}_mixture_{MIXTURE_SIZE}.png", dpi=150)
plt.show()


## Next steps

**Next:** `08_closed_form_calibration.ipynb` — the NRC-weighted correction.
Its design (which NRC component to weight by, whether to target
`pce_base` or `pce_qrc`) should be informed by which correlation from `06`
turned out strongest **on real data**, not assumed in advance — revisit
after running `00`-`07` for real on Colab.
